In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!uv pip install lightgbm scikit-learn pandas pyarrow matplotlib ucimlrepo --quiet

In [ ]:
import lightgbm as lgbm
from pomap.interface import leaf, feed, ensemble

import polars as pl

from functools import partial
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

In [ ]:
data = pl.from_pandas(fetch_ucirepo(id=275).data.original)

data = data.with_columns(dteday=pl.col("dteday").str.to_datetime())

In [ ]:
class LGBMWrapper:
    def __init__(self, features, target, **kwargs):
        self.model = lgbm.LGBMRegressor(**kwargs)
        self.features = features
        self.target = target

    def fit(self, training_set: pl.DataFrame, validation_set: pl.DataFrame = None):

        X_train = training_set.select(*self.features).to_pandas()
        y_train = training_set[self.target].to_pandas()

        eval_set = [(X_train, y_train)]
        eval_names = ["training"]

        if validation_set is not None:
            X_val = validation_set.select(*self.features).to_pandas()
            y_val = validation_set[self.target].to_pandas()
            eval_set += [(X_val, y_val)]
            eval_names += ["validation"]

        self.model.fit(X_train, y_train, eval_set=eval_set, eval_names=eval_names)

    def predict(self, df: pl.DataFrame):
        X = df.select(*self.features).to_pandas()
        return self.model.predict(X)

In [ ]:
features = [
    "hr",
    "weekday",
    "workingday",
    "holiday",
    "season",
    "mnth",
    "temp",
    "hum",
    "windspeed",
    "weathersit",
]

target = "cnt"

In [ ]:
teacher_model = partial(
    LGBMWrapper,
    features=features,
    target=target,
    num_leaves=31,
    max_depth=5,
    learning_rate=0.01,
    n_estimators=10000,
)

student_model = partial(
    LGBMWrapper,
    features=features,
    target="teacher",
    num_leaves=31,
    max_depth=5,
    learning_rate=0.01,
    n_estimators=1000,
)

untutored_student = partial(
    LGBMWrapper,
    features=features,
    target=target,
    num_leaves=31,
    max_depth=5,
    learning_rate=0.01,
    n_estimators=1000,
)

In [ ]:
teacher = leaf(teacher_model, "teacher")
student = leaf(student_model, "student")
untutored_student = leaf(untutored_student, "untutored_student")

In [ ]:
distillation = feed("distillation", source=teacher, consumer=student)
full_experiment = ensemble("full", distillation, untutored_student)

In [ ]:
full_experiment.fit(data);

In [ ]:
fig, ax = plt.subplots()

for label, model in full_experiment.models.items():
    ax.plot(model.model.evals_result_["training"]["l2"])

In [ ]:
for label, model in full_experiment.models.items():
    print(label)
    print(model.model.best_score_["training"]["l2"])

In [ ]:
fitted = full_experiment.predict(data)